# LSTM on the MALL 5-min Feature Set


**Purpose:** The purpose of this notebook is to compute the out-of-sample forecast performance of an `LSTM` model on the $\mathcal{M}_{\mathrm{ALL}}$ 5-min feature set for the EURO STOXX 50 index over the period 1999--2020. This model is implemented as a thesis extension to Christensen et al. (2023), while keeping the surrounding estimation workflow as close as possible to the neural-network notebooks already used in this project.

This initial implementation uses a compact pilot specification to keep runtime manageable while still giving the LSTM a genuine sequential structure:

- final MALL predictors:
  - `RVD`, `RVW`, `RVM`
  - `VSTOXX`
  - `CISS`
  - `logEPU_1m_lag`
  - `EURUSD_ret`
  - `BRENT_ret`
  - `NIKKEI_ret`, `NIKKEI_sq`
  - `SP500_ret_lag`, `SP500_sq_lag`
  - `M1W`, `M1M`
- input sequences of the previous `10` trading days
- `1` LSTM layer with hidden size `16`
- dropout rate `0.2` applied to the final hidden state before the output layer
- Adam optimizer with learning rate `0.001`
- maximum `300` epochs and early stopping patience `50`
- `10` independent seed initializations ranked by validation MSE
- forecasts from:
  - the single best network (`ensemble size = 1`)
  - the average of the top `10` networks (`ensemble size = 10`)

The sample split remains fixed at `70 / 10 / 20`, matching the existing MALL neural-network notebooks. Predictors and the target variable are standardized using training-sample moments only, and forecasts are back-transformed to the original realized-variance scale for evaluation.


## 1. Loading Data
Loading in data from the `MALL_5min.csv` file.


In [4]:
import os
import copy
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

# Load MALL dataset
file_path = "/Users/tobiasbergdahlpersson/Documents/SSE/MSc Thesis/5-min RV/Cleaned Data Sets/MALL_5min.csv"
MALL = pd.read_csv(file_path, index_col="Date", parse_dates=True)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Shape: {MALL.shape}")
print(f"Date range: {MALL.index[0].date()} to {MALL.index[-1].date()}")
print(f"NaN values: {MALL.isna().sum().sum()}")
MALL.head()


Using device: cpu
Shape: (5489, 31)
Date range: 1999-01-05 to 2020-08-06
NaN values: 0


,RVD,RVW,RVM,logRVD,logRVW,logRVM,RVD_pos,RVD_neg,RD_neg,RW_neg,...,BRENT,BRENT_ret,NIKKEI_ret,NIKKEI_sq,SP500_ret_lag,SP500_sq_lag,M1W,M1M,RV_target,logRV_target
Date,,,,,,,,,,,,,,,,,,,,,
1999-01-05,0.000106,0.000089,0.000141,-9.153100,-9.328531,-8.866447,0.000071,0.000035,0.000000,0.000000,...,10.30,-0.060282,-0.013746,0.000189,-0.000920,8.458434e-07,0.087690,0.126206,0.000047,-9.966398
1999-01-06,0.000047,0.000090,0.000132,-9.966398,-9.316927,-8.935760,0.000038,0.000009,0.000000,0.000000,...,10.67,0.035292,0.017657,0.000312,0.013491,1.819949e-04,0.100143,0.187219,0.000237,-8.347779
1999-01-07,0.000237,0.000123,0.000131,-8.347779,-9.004761,-8.938374,0.000110,0.000127,-0.012859,0.000000,...,11.08,0.037706,0.005044,0.000025,0.021899,4.795626e-04,0.078881,0.184020,0.000168,-8.689606
1999-01-08,0.000168,0.000146,0.000114,-8.689606,-8.832472,-9.082830,0.000079,0.000089,-0.005883,0.000000,...,11.70,0.054447,-0.010751,0.000116,-0.002053,4.216638e-06,0.078883,0.165510,0.000282,-8.175339
1999-01-11,0.000282,0.000168,0.000117,-8.175339,-8.692020,-9.056351,0.000067,0.000215,-0.025309,-0.001398,...,12.07,0.031134,-0.001744,0.000003,0.004212,1.774503e-05,0.001055,0.140613,0.000191,-8.565468


## 2. Data Split
### 2.1 Train / Validation / Test Split (70 / 10 / 20)


Following Christensen et al. (2023), the sample is split chronologically into a training set (70%), validation set (10%), and test set (20%). The temporal ordering is strictly preserved to avoid look-ahead bias. For this LSTM extension, the split remains fixed throughout the out-of-sample evaluation, matching the existing MALL neural-network notebooks.


In [8]:
# Train / Validation / Test split (70 / 10 / 20)
n = len(MALL)
n_train = int(np.floor(0.70 * n))
n_val = int(np.floor(0.10 * n))
n_test = n - n_train - n_val

train_idx = MALL.index[:n_train]
val_idx = MALL.index[n_train:n_train + n_val]
test_idx = MALL.index[n_train + n_val:]

print(f"Total observations: {n}")
print(f"\nTrain:      {train_idx[0].date()} to {train_idx[-1].date()} ({n_train} obs, {n_train/n*100:.1f}%)")
print(f"Validation: {val_idx[0].date()} to {val_idx[-1].date()} ({n_val} obs, {n_val/n*100:.1f}%)")
print(f"Test:       {test_idx[0].date()} to {test_idx[-1].date()} ({n_test} obs, {n_test/n*100:.1f}%)")


Total observations: 5489

Train:      1999-01-05 to 2014-02-26 (3842 obs, 70.0%)
Validation: 2014-02-27 to 2016-04-20 (548 obs, 10.0%)
Test:       2016-04-21 to 2020-08-06 (1099 obs, 20.0%)


## 3. MALL Feature Set Construction, Standardization, and Sequence Building

The `LSTM` model is estimated on the final MALL feature set, consisting of the MHAR realized-volatility block augmented by macro-financial and cross-market predictors. As in the MALL feedforward neural-network notebooks, the explanatory variables are standardized using training-sample moments only, and the target is also standardized for numerical stability.

Because an LSTM requires sequential input, we convert the standardized MALL rows into rolling input windows. In this pilot specification, each forecast is based on the previous `10` trading days of MALL inputs. Validation and test sequences are allowed to draw on past observations from the preceding split, which preserves chronology and does not introduce look-ahead bias.


In [11]:
# Define target, final MALL feature set, and LSTM lookback
feature_cols = [
    "RVD", "RVW", "RVM",
    "VSTOXX", "CISS", "logEPU_1m_lag",
    "EURUSD_ret", "BRENT_ret",
    "NIKKEI_ret", "NIKKEI_sq",
    "SP500_ret_lag", "SP500_sq_lag",
    "M1W", "M1M"
]
LOOKBACK = 10

y = MALL["RV_target"].values
X_MALL = MALL[feature_cols].values

# Raw splits used for scaling moments
X_train_raw = X_MALL[:n_train]
y_train_raw = y[:n_train]

# Feature scaling using training moments only
X_mean = X_train_raw.mean(axis=0)
X_std = X_train_raw.std(axis=0, ddof=0)
X_std = np.where(X_std == 0, 1.0, X_std)
X_scaled = (X_MALL - X_mean) / X_std

# Target scaling using training moments only
y_mean = y_train_raw.mean()
y_std = y_train_raw.std(ddof=0)
if y_std == 0:
    y_std = 1.0
y_scaled = (y - y_mean) / y_std

def create_sequences(X_all, y_all_scaled, y_all_raw, dates, start_idx, end_idx, lookback):
    X_seq = []
    y_seq_scaled = []
    y_seq_raw = []
    seq_dates = []

    for t in range(start_idx, end_idx):
        if t - lookback < 0:
            continue
        X_seq.append(X_all[t - lookback:t])
        y_seq_scaled.append(y_all_scaled[t])
        y_seq_raw.append(y_all_raw[t])
        seq_dates.append(dates[t])

    return (
        np.asarray(X_seq, dtype=np.float32),
        np.asarray(y_seq_scaled, dtype=np.float32),
        np.asarray(y_seq_raw, dtype=np.float64),
        pd.Index(seq_dates)
    )

X_train_seq, y_train_seq_scaled, y_train_seq_raw, train_seq_idx = create_sequences(
    X_scaled, y_scaled, y, MALL.index, LOOKBACK, n_train, LOOKBACK
)
X_val_seq, y_val_seq_scaled, y_val_seq_raw, val_seq_idx = create_sequences(
    X_scaled, y_scaled, y, MALL.index, n_train, n_train + n_val, LOOKBACK
)
X_test_seq, y_test_seq_scaled, y_test_seq_raw, test_seq_idx = create_sequences(
    X_scaled, y_scaled, y, MALL.index, n_train + n_val, n, LOOKBACK
)

print("Feature set:")
for col in feature_cols:
    print(f"  - {col}")
print(f"\nLSTM lookback: {LOOKBACK} trading days")
print(f"Features per step: {len(feature_cols)}")

print(f"\nTraining sequences:   X={X_train_seq.shape}, y={y_train_seq_scaled.shape}")
print(f"Validation sequences: X={X_val_seq.shape}, y={y_val_seq_scaled.shape}")
print(f"Test sequences:       X={X_test_seq.shape}, y={y_test_seq_scaled.shape}")

print("\nTraining target dates available to the LSTM:")
print(f"  Train: {train_seq_idx[0].date()} to {train_seq_idx[-1].date()}")
print(f"  Validation: {val_seq_idx[0].date()} to {val_seq_idx[-1].date()}")
print(f"  Test: {test_seq_idx[0].date()} to {test_seq_idx[-1].date()}")

print("\nPredictor standardization check:")
print(f"  Means (first 5): {((X_train_raw - X_mean) / X_std).mean(axis=0)[:5].round(6)}")
print(f"  Std   (first 5): {((X_train_raw - X_mean) / X_std).std(axis=0)[:5].round(6)}")
print("\nTarget standardization check:")
print(f"  Train mean (scaled): {y_scaled[:n_train].mean():.6f}")
print(f"  Train std  (scaled): {y_scaled[:n_train].std():.6f}")
print(f"  Raw target mean:     {y_mean:.6e}")
print(f"  Raw target std:      {y_std:.6e}")


Feature set:
  - RVD
  - RVW
  - RVM
  - VSTOXX
  - CISS
  - logEPU_1m_lag
  - EURUSD_ret
  - BRENT_ret
  - NIKKEI_ret
  - NIKKEI_sq
  - SP500_ret_lag
  - SP500_sq_lag
  - M1W
  - M1M

LSTM lookback: 10 trading days
Features per step: 14

Training sequences:   X=(3832, 10, 14), y=(3832,)
Validation sequences: X=(548, 10, 14), y=(548,)
Test sequences:       X=(1099, 10, 14), y=(1099,)

Training target dates available to the LSTM:
  Train: 1999-01-19 to 2014-02-26
  Validation: 2014-02-27 to 2016-04-20
  Test: 2016-04-21 to 2020-08-06

Predictor standardization check:
  Means (first 5): [ 0. -0. -0. -0.  0.]
  Std   (first 5): [1. 1. 1. 1. 1.]

Target standardization check:
  Train mean (scaled): 0.000000
  Train std  (scaled): 1.000000
  Raw target mean:     1.608657e-04
  Raw target std:      2.992674e-04


## 4. Define the LSTM Architecture

The model uses a single-layer LSTM with hidden size `16`. Each input sequence contains the previous `10` trading days of the standardized MALL predictors. The final hidden state is passed through dropout with rate `0.2` and then mapped to a one-dimensional forecast via a linear output layer.


In [14]:
class LSTMForecaster(nn.Module):
    def __init__(self, input_dim, hidden_size=16, num_layers=1, dropout_rate=0.2):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )
        self.dropout = nn.Dropout(p=dropout_rate)
        self.output = nn.Linear(hidden_size, 1)
        self._initialize_weights()

    def _initialize_weights(self):
        for name, param in self.lstm.named_parameters():
            if "weight_ih" in name:
                nn.init.xavier_normal_(param)
            elif "weight_hh" in name:
                nn.init.orthogonal_(param)
            elif "bias" in name:
                nn.init.zeros_(param)
                hidden_size = param.shape[0] // 4
                param.data[hidden_size:2 * hidden_size].fill_(1.0)

        nn.init.xavier_normal_(self.output.weight)
        nn.init.zeros_(self.output.bias)

    def forward(self, x):
        _, (hidden_n, _) = self.lstm(x)
        last_hidden = hidden_n[-1]
        last_hidden = self.dropout(last_hidden)
        out = self.output(last_hidden)
        return out.squeeze(-1)

INPUT_DIM = X_train_seq.shape[2]
HIDDEN_SIZE = 16
NUM_LAYERS = 1
DROPOUT_RATE = 0.2

model_test = LSTMForecaster(
    input_dim=INPUT_DIM,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    dropout_rate=DROPOUT_RATE
).to(device)

print(model_test)
print(f"\nSequence length:    {LOOKBACK}")
print(f"Features per step:  {INPUT_DIM}")
print(f"Hidden size:        {HIDDEN_SIZE}")
print(f"LSTM layers:        {NUM_LAYERS}")
print(f"Dropout rate:       {DROPOUT_RATE}")
print("Output dimension:   1")
print(f"\nTotal parameters: {sum(p.numel() for p in model_test.parameters())}")


LSTMForecaster(
  (lstm): LSTM(14, 16, batch_first=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (output): Linear(in_features=16, out_features=1, bias=True)
)

Sequence length:    10
Features per step:  14
Hidden size:        16
LSTM layers:        1
Dropout rate:       0.2
Output dimension:   1

Total parameters: 2065


## 5. Training Function with Early Stopping

The LSTM is trained by minimizing mean squared error in standardized target space using Adam with learning rate `0.001`. To keep the first run computationally manageable, this pilot specification uses at most `300` epochs and early stopping with patience `50`. The best validation-state parameters are restored at the end of training.


In [17]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def to_tensor(x):
    return torch.tensor(x, dtype=torch.float32, device=device)


def train_lstm(X_tr, y_tr, X_vl, y_vl,
               input_dim,
               hidden_size=16,
               num_layers=1,
               dropout_rate=0.2,
               learning_rate=0.001,
               max_epochs=300,
               patience=50,
               seed=0):

    set_seed(seed)

    X_tr_t = to_tensor(X_tr)
    y_tr_t = to_tensor(y_tr)
    X_vl_t = to_tensor(X_vl)
    y_vl_t = to_tensor(y_vl)

    model = LSTMForecaster(
        input_dim=input_dim,
        hidden_size=hidden_size,
        num_layers=num_layers,
        dropout_rate=dropout_rate
    ).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    best_val_loss = np.inf
    best_state = None
    best_epoch = 0
    epochs_no_improve = 0

    for epoch in range(max_epochs):
        model.train()
        optimizer.zero_grad()
        pred_tr = model(X_tr_t)
        loss_tr = criterion(pred_tr, y_tr_t)
        loss_tr.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            pred_vl = model(X_vl_t)
            loss_vl = criterion(pred_vl, y_vl_t).item()

        if loss_vl < best_val_loss:
            best_val_loss = loss_vl
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            break

    model.load_state_dict(best_state)
    model.eval()

    return model, best_val_loss, best_epoch


## 6. Initial Validation Check

Before estimating the seed ensemble, we train a single LSTM on the initial training / validation split as a sanity check. The validation loss is reported in standardized target space, while predictions are back-transformed to the original realized-variance scale for interpretation.


In [20]:
model_check, val_loss_check, best_epoch_check = train_lstm(
    X_tr=X_train_seq,
    y_tr=y_train_seq_scaled,
    X_vl=X_val_seq,
    y_vl=y_val_seq_scaled,
    input_dim=INPUT_DIM,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    dropout_rate=DROPOUT_RATE,
    seed=0
)

with torch.no_grad():
    preds_check_scaled = model_check(to_tensor(X_val_seq)).detach().cpu().numpy()
preds_check = preds_check_scaled * y_std + y_mean

print(f"Validation MSE (scaled, seed=0): {val_loss_check:.6e}")
print(f"Best epoch:                      {best_epoch_check}")
print(f"Mean validation forecast:       {preds_check.mean():.6e}")
print(f"Mean actual validation RV:      {y_val_seq_raw.mean():.6e}")
print(f"Any negative forecasts:         {(preds_check < 0).sum()}")
print(f"Any NaN forecasts:              {np.isnan(preds_check).sum()}")


Validation MSE (scaled, seed=0): 1.272832e-01
Best epoch:                      31
Mean validation forecast:       1.214210e-04
Mean actual validation RV:      1.221296e-04
Any negative forecasts:         0
Any NaN forecasts:              0


## 7. Ensemble Training on the Fixed Estimation Window

For this first implementation, we train `10` independent versions of the same `LSTM` architecture with different random seeds and rank them by validation MSE in standardized target space. We then retain:

- the single best network (`ensemble size = 1`)
- the top `10` networks (`ensemble size = 10`)

Because only `10` seeds are trained in this pilot run, the top-10 ensemble coincides with averaging all trained networks. If the LSTM looks promising, this can later be scaled up to a larger seed search.


In [23]:
N_SEEDS = 10
TOP_K = min(10, N_SEEDS)

seed_records = []
start_time = time.time()

print(f"Training {N_SEEDS} independent LSTM networks...")
for seed in range(N_SEEDS):
    model_i, val_loss_i, best_epoch_i = train_lstm(
        X_tr=X_train_seq,
        y_tr=y_train_seq_scaled,
        X_vl=X_val_seq,
        y_vl=y_val_seq_scaled,
        input_dim=INPUT_DIM,
        hidden_size=HIDDEN_SIZE,
        num_layers=NUM_LAYERS,
        dropout_rate=DROPOUT_RATE,
        seed=seed
    )
    seed_records.append({
        "seed": seed,
        "val_loss": val_loss_i,
        "best_epoch": best_epoch_i,
        "model": model_i
    })

    elapsed = (time.time() - start_time) / 60
    print(f"  Completed {seed + 1:2d}/{N_SEEDS} seeds | Elapsed: {elapsed:.2f} min")

elapsed = (time.time() - start_time) / 60
print(f"\nTraining time: {elapsed:.2f} minutes")

# Rank by validation loss
seed_records.sort(key=lambda x: x["val_loss"])
best_network = seed_records[0]
top10_networks = seed_records[:TOP_K]

ranking_df = pd.DataFrame([
    {
        "rank": i + 1,
        "seed": rec["seed"],
        "val_mse_scaled": rec["val_loss"],
        "best_epoch": rec["best_epoch"]
    }
    for i, rec in enumerate(seed_records)
])

print("\nTop seeds by validation MSE:")
display(ranking_df.head(TOP_K))


Training 10 independent LSTM networks...
  Completed  1/10 seeds | Elapsed: 0.02 min
  Completed  2/10 seeds | Elapsed: 0.07 min
  Completed  3/10 seeds | Elapsed: 0.14 min
  Completed  4/10 seeds | Elapsed: 0.22 min
  Completed  5/10 seeds | Elapsed: 0.24 min
  Completed  6/10 seeds | Elapsed: 0.26 min
  Completed  7/10 seeds | Elapsed: 0.27 min
  Completed  8/10 seeds | Elapsed: 0.34 min
  Completed  9/10 seeds | Elapsed: 0.42 min
  Completed 10/10 seeds | Elapsed: 0.44 min

Training time: 0.44 minutes

Top seeds by validation MSE:


,rank,seed,val_mse_scaled,best_epoch
0,1,0,0.127283,31
1,2,8,0.130040,266
2,3,2,0.131245,218
3,4,7,0.134032,219
4,5,3,0.137825,300
5,6,4,0.138009,25
6,7,1,0.138821,107
7,8,6,0.141399,18
8,9,9,0.145173,43
9,10,5,0.161555,28


## 8. One-Day-Ahead Test Forecasts

The selected `LSTM` networks are now used to forecast the entire test set. The forecasts are first produced in standardized target space and then back-transformed to the original realized-variance scale. If any forecast is negative, it is replaced by the minimum realized variance observed in the pre-test sample (training plus validation), which acts as the same non-negativity safeguard used in the MALL neural-network notebooks.


In [26]:
X_test_t = to_tensor(X_test_seq)
min_rv_insample = y[:n_train + n_val].min()

# Best network forecast
best_network["model"].eval()
with torch.no_grad():
    pred_best_scaled = best_network["model"](X_test_t).detach().cpu().numpy()
pred_best = pred_best_scaled * y_std + y_mean
pred_best = np.maximum(pred_best, min_rv_insample)

# Top-10 ensemble forecast
pred_top10_all = []
for rec in top10_networks:
    rec["model"].eval()
    with torch.no_grad():
        pred_i_scaled = rec["model"](X_test_t).detach().cpu().numpy()
        pred_i = pred_i_scaled * y_std + y_mean
        pred_top10_all.append(pred_i)

pred_top10 = np.mean(pred_top10_all, axis=0)
pred_top10 = np.maximum(pred_top10, min_rv_insample)

results_LSTM = pd.DataFrame({
    "RV_actual": y_test_seq_raw,
    "RV_forecast_best": pred_best,
    "RV_forecast_top10": pred_top10
}, index=test_seq_idx)

errors_best = results_LSTM["RV_actual"] - results_LSTM["RV_forecast_best"]
errors_top10 = results_LSTM["RV_actual"] - results_LSTM["RV_forecast_top10"]

mse_best = np.mean(errors_best ** 2)
mse_top10 = np.mean(errors_top10 ** 2)

print(f"Forecast period: {test_seq_idx[0].date()} to {test_seq_idx[-1].date()}")
print(f"In-sample floor: {min_rv_insample:.6e}")
print(f"\nLSTM_best MSE: {mse_best:.6e}")
print(f"LSTM_top10 MSE: {mse_top10:.6e}")


Forecast period: 2016-04-21 to 2020-08-06
In-sample floor: 3.910328e-08

LSTM_best MSE: 7.651218e-08
LSTM_top10 MSE: 6.317902e-08


## 9. Save Forecasts and Summary Statistics


In [ ]:
forecast_path = "/Users/tobiasbergdahlpersson/Documents/SSE/MSc Thesis/5-min RV/Cleaned Data Sets/Forecasts 5-min/MALL/Neural Networks"
mse_path = "/Users/tobiasbergdahlpersson/Documents/SSE/MSc Thesis/5-min RV/Cleaned Data Sets/MSE 5-min/MALL/Neural Networks"

os.makedirs(forecast_path, exist_ok=True)
os.makedirs(mse_path, exist_ok=True)

results_LSTM.to_csv(os.path.join(forecast_path, "forecasts_LSTM.csv"))
mse_summary_LSTM = pd.DataFrame({
    "Model": ["LSTM_best", "LSTM_top10"],
    "MSE": [mse_best, mse_top10],
    "RMSE": [np.sqrt(mse_best), np.sqrt(mse_top10)],
    "MAE": [errors_best.abs().mean(), errors_top10.abs().mean()],
    "Mean_Error": [errors_best.mean(), errors_top10.mean()]
}).set_index("Model")

mse_summary_LSTM.to_csv(os.path.join(mse_path, "mse_summary_LSTM.csv"))

print(f"Forecasts saved to:   {forecast_path}")
print(f"MSE summary saved to: {mse_path}")
print(f"Rows saved: {len(results_LSTM)}")
print()
print(mse_summary_LSTM.to_string())


## 10. Results
### 10.1 Forecast Visualization


In [ ]:
results_plot = pd.DataFrame({
    "Actual": np.sqrt(results_LSTM["RV_actual"] * 252) * 100,
    "Forecast_best": np.sqrt(results_LSTM["RV_forecast_best"] * 252) * 100,
    "Forecast_top10": np.sqrt(results_LSTM["RV_forecast_top10"] * 252) * 100
}, index=results_LSTM.index)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(results_plot["Actual"], label="Actual RV", color="black", linewidth=0.8)
ax.plot(results_plot["Forecast_best"], label="LSTM Best Forecast", color="red", linewidth=0.8, linestyle="--")
ax.plot(results_plot["Forecast_top10"], label="LSTM Top-10 Ensemble", color="blue", linewidth=0.8, linestyle=":")
ax.set_title("LSTM - One-Day-Ahead Realized Volatility Forecast\nEURO STOXX 50 (2016-2020)", fontsize=12)
ax.set_ylabel("Annualized Volatility (%)")
ax.set_xlabel("Date")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nSummary of forecast errors:")
print("\nLSTM Best:")
print(f"  Mean error:  {errors_best.mean():.6e}")
print(f"  RMSE:        {np.sqrt(mse_best):.6e}")
print(f"  MAE:         {errors_best.abs().mean():.6e}")

print("\nLSTM Top-10 Ensemble:")
print(f"  Mean error:  {errors_top10.mean():.6e}")
print(f"  RMSE:        {np.sqrt(mse_top10):.6e}")
print(f"  MAE:         {errors_top10.abs().mean():.6e}")
